# درس ۰۴ - الگوی طراحی استفاده از ابزار

در این درس الگوی طراحی **استفاده از ابزار** برای عوامل هوش مصنوعی با استفاده از چارچوب Microsoft Agent (پایتون) را خواهید آموخت. موارد زیر پوشش داده می‌شود:

- تعریف توابع ابزار با دکوراتور `@tool` و پارامترهای تایپ شده
- ارائه طرح‌واره‌های ابزار تا مدل بداند هر ابزار چه کاری انجام می‌دهد
- کنترل اجرای ابزار با `approval_mode`
- بازگرداندن **خروجی ساختاریافته** از طریق مدل‌های Pydantic و `response_format`

سناریو یک **عامل رزرو سفر** است که می‌تواند مقصدها را جستجو کند، در دسترس بودن را بررسی کند و اطلاعات پرواز را بازیابی کند.


## راه‌اندازی


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## تعریف ابزارها با دکوراتور @tool

دکوراتور `@tool` یک تابع ساده پایتون را به ابزاری تبدیل می‌کند که یک عامل می‌تواند فراخوانی کند.  
نکات کلیدی:

- **داکیوسترینگ** به توضیح ابزار تبدیل می‌شود که مدل آن را می‌بیند.  
- **توضیحات نوع** (شامل `Annotated` با توضیحات) طرح ابزار را تعریف می‌کنند.  
- `approval_mode` کنترل می‌کند که آیا کاربر باید هر فراخوانی را قبل از اجرا تأیید کند یا خیر.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ایجاد یک عامل با ابزارهای متعدد

تمامی سه ابزار را به مشتری منتقل کنید تا مدل بتواند هرکدام را که برای پاسخ به سوال کاربر لازم است فراخوانی کند.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## خروجی ساختاریافته با ابزارها

با تنظیم `response_format` به یک مدل Pydantic، عامل مجبور است یک شیء JSON با نوع‌بندی دقیق را به جای متن آزاد بازگرداند. این زمانی مفید است که کد پایین‌دستی نیاز دارد نتیجه را به صورت برنامه‌نویسی مصرف کند.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## الگوهای تایید ابزار

پارامتر `approval_mode` در `@tool` کنترل می‌کند که آیا فراخوانی‌های ابزار قبل از اجرا نیاز به تایید انسانی دارند یا خیر:

| حالت | رفتار |
|---|---|
| `"never_require"` | ابزار به‌صورت خودکار اجرا می‌شود — نیازی به تایید کاربر ندارد. |
| `"always_require"` | هر فراخوانی قبل از اجرا باید توسط کاربر تایید شود. |

از `"always_require"` برای ابزارهایی استفاده کنید که اثرات جانبی دارند (مثلاً رزرو پرواز، شارژ کارت اعتباری) تا یک انسان در روند کار حضور داشته باشد.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## خلاصه

در این درس آموختید چگونه:

1. **تعریف ابزارها** با استفاده از دکوراتور `@tool` همراه با پارامترهای تایپ شده و docstringهایی که به عنوان طرح ابزار عمل می‌کنند.
2. **ترکیب چند ابزار** به طوری که نماینده بتواند آنها را به ترتیب فراخوانی کند تا به پرسش‌های پیچیده پاسخ دهد.
3. **بازگرداندن خروجی ساختاریافته** با ارسال یک مدل Pydantic به عنوان `response_format`.
4. **کنترل تأیید ابزار** با `approval_mode` برای حفظ حضور یک انسان در فرایند برای عملیات حساس.

این الگوها پایه‌ای برای ساخت نمایندگان قابل اعتماد و آماده تولید است که می‌توانند به صورت ایمن با سیستم‌های خارجی تعامل داشته باشند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
